# 04 Methodology, Results, and Diagnostics

This notebook matches the "current pipeline + workbook contract" style used in notebooks 01–03:

- Load the processed panel contract (`data/processed/clean_panel.csv`)
- Assert the active window (`2000–2023`)
- Build the **workbook-driven model catalog** (source of truth for which models exist / are estimated)
- Load exported estimation artifacts produced by `02_panel_diagnostics.ipynb` (if present)
- Save a small set of deterministic **CSV + PNG** artifacts under `outputs/`


## Setup


In [91]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.config as config
import src.model_contract as model_contract
import src.reporting as reporting

importlib.reload(config)  # stale-kernel guard
importlib.reload(model_contract)
importlib.reload(reporting)

OUTPUTS_DIR = config.OUTPUTS_DIR
FIGURES_DIR = config.FIGURES_DIR
PROCESSED_PANEL_FILE = config.PROCESSED_PANEL_FILE
TIME_WINDOW = config.TIME_WINDOW
ensure_output_dirs = config.ensure_output_dirs
compact_display = reporting.compact_display
VARIABLE_LABELS = reporting.VARIABLE_LABELS

ensure_output_dirs()
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")


## Load Processed Panel (New Data Contract)

Source of truth: `data/processed/clean_panel.csv` produced by `01_data_processing.ipynb`.


In [92]:
df = pd.read_csv(PROCESSED_PANEL_FILE)
df.shape, int(df['year'].min()), int(df['year'].max())


((264, 18), 2000, 2023)

In [93]:
start_year, end_year = TIME_WINDOW
year_min, year_max = int(df['year'].min()), int(df['year'].max())
if (year_min, year_max) != (start_year, end_year):
    raise ValueError(f'Expected year window {TIME_WINDOW}, got {(year_min, year_max)}. Re-run 01_data_processing.ipynb.')

snapshot_vars = [
    'fdi_pct_gdp',
    'broad_money_growth_pct',
    'inflation_gdp_deflator_pct',
    'trade_pct_gdp',
    'ln_gdppc',
    'xr_dep_pct',
    'real_interest_rate_pct',
    'deposit_interest_rate_pct',
    'lending_interest_rate_pct',
    'ln_tourism_arrivals',
    'hc_human_capital_index',
]
snapshot_vars = [c for c in snapshot_vars if c in df.columns]
coverage = (
    df[snapshot_vars]
    .notna()
    .sum()
    .rename('non_missing')
    .to_frame()
    .assign(non_missing_pct=lambda frame: 100 * frame['non_missing'] / len(df))
    .reset_index()
    .rename(columns={'index': 'variable'})
    .sort_values(['non_missing', 'variable'], ascending=[True, True])
    .reset_index(drop=True)
)
coverage


,variable,non_missing,non_missing_pct
0,lending_interest_rate_pct,192,72.7273
1,real_interest_rate_pct,205,77.6515
2,deposit_interest_rate_pct,213,80.6818
3,broad_money_growth_pct,240,90.9091
4,hc_human_capital_index,240,90.9091
5,ln_tourism_arrivals,240,90.9091
6,trade_pct_gdp,240,90.9091
7,xr_dep_pct,253,95.8333
8,fdi_pct_gdp,260,98.4848
9,inflation_gdp_deflator_pct,264,100.0000


## Workbook Model Contract (Source of Truth)

The workbook (`model_selection_asean_fdi.xlsx`) defines which specifications exist and which ones can be estimated
given the processed panel columns.


In [94]:
catalog, workbook_tables = model_contract.build_workbook_model_catalog(
    config.MODEL_SELECTION_WORKBOOK,
    panel_columns=df.columns.tolist(),
)

view_cols = [
    'model_id',
    'workbook_code',
    'workbook_model',
    'status',
    'missing_reason',
    'lagged_model',
    'mapped_regressors',
]
compact_display(catalog, columns=view_cols, n=30)

estimated_catalog = catalog[catalog['status'].eq('estimated')].copy()
estimated_model_ids = estimated_catalog['model_id'].tolist()
print('Estimated models:', len(estimated_model_ids))
estimated_catalog[['model_id', 'workbook_code', 'workbook_model', 'mapped_regressors']].head(10)


,model_id,workbook_code,workbook_model,status,missing_reason,lagged_model,mapped_regressors
0,M1_baseline_liquidity,M1,M1 - Baseline liquidity,estimated,,False,"broad_money_growth_pct, inflation_gdp_deflator..."
1,M2_main_monetary_policy,M2,M2 - Main monetary policy,estimated,,False,"broad_money_growth_pct, deposit_interest_rate_..."
2,M3_lagged_main_model,M3,M3 - Lagged main model,estimated,,True,"broad_money_growth_pct_lag1, deposit_interest_..."
3,M4_real_interest_robustness,M4,M4 - Real interest robustness,estimated,,False,"broad_money_growth_pct, real_interest_rate_pct..."
4,M5_lending_rate_robustness,M5,M5 - Lending rate robustness,estimated,,False,"broad_money_growth_pct, lending_interest_rate_..."
5,M6_tourism_robustnessa_from_M4_real_interest_r...,M6,M6 - Tourism robustness,estimated,,False,"broad_money_growth_pct, real_interest_rate_pct..."
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,M6,M6 - Tourism robustness,estimated,,False,"broad_money_growth_pct, deposit_interest_rate_..."
7,M7_human_capital_robustnessa_from_M4_real_inte...,M7,M7 - Human capital robustness,estimated,,False,"broad_money_growth_pct, real_interest_rate_pct..."
8,M7_human_capital_robustnessb_from_M2_main_mone...,M7,M7 - Human capital robustness,estimated,,False,"broad_money_growth_pct, deposit_interest_rate_..."


Estimated models: 9


,model_id,workbook_code,workbook_model,mapped_regressors
0,M1_baseline_liquidity,M1,M1 - Baseline liquidity,"broad_money_growth_pct, inflation_gdp_deflator..."
1,M2_main_monetary_policy,M2,M2 - Main monetary policy,"broad_money_growth_pct, deposit_interest_rate_..."
2,M3_lagged_main_model,M3,M3 - Lagged main model,"broad_money_growth_pct_lag1, deposit_interest_..."
3,M4_real_interest_robustness,M4,M4 - Real interest robustness,"broad_money_growth_pct, real_interest_rate_pct..."
4,M5_lending_rate_robustness,M5,M5 - Lending rate robustness,"broad_money_growth_pct, lending_interest_rate_..."
5,M6_tourism_robustnessa_from_M4_real_interest_r...,M6,M6 - Tourism robustness,"broad_money_growth_pct, real_interest_rate_pct..."
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,M6,M6 - Tourism robustness,"broad_money_growth_pct, deposit_interest_rate_..."
7,M7_human_capital_robustnessa_from_M4_real_inte...,M7,M7 - Human capital robustness,"broad_money_growth_pct, real_interest_rate_pct..."
8,M7_human_capital_robustnessb_from_M2_main_mone...,M7,M7 - Human capital robustness,"broad_money_growth_pct, deposit_interest_rate_..."


In [95]:
# Prefer module outputs written by notebook 02 (source of truth on disk).
workbook_catalog_path = OUTPUTS_DIR / 'workbook_model_catalog.csv'
if workbook_catalog_path.exists():
    workbook_catalog_disk = pd.read_csv(workbook_catalog_path)
    compact_display(workbook_catalog_disk, columns=view_cols, n=30)
else:
    print('MISSING:', workbook_catalog_path.name, '(run 02_panel_diagnostics.ipynb)')


,model_id,workbook_code,workbook_model,status,missing_reason,lagged_model,mapped_regressors
0,M1_baseline_liquidity,M1,M1 - Baseline liquidity,estimated,NaN,False,"broad_money_growth_pct, inflation_gdp_deflator..."
1,M2_main_monetary_policy,M2,M2 - Main monetary policy,estimated,NaN,False,"broad_money_growth_pct, deposit_interest_rate_..."
2,M3_lagged_main_model,M3,M3 - Lagged main model,estimated,NaN,True,"broad_money_growth_pct_lag1, deposit_interest_..."
3,M4_real_interest_robustness,M4,M4 - Real interest robustness,estimated,NaN,False,"broad_money_growth_pct, real_interest_rate_pct..."
4,M5_lending_rate_robustness,M5,M5 - Lending rate robustness,estimated,NaN,False,"broad_money_growth_pct, lending_interest_rate_..."
5,M6_tourism_robustnessa_from_M4_real_interest_r...,M6,M6 - Tourism robustness,estimated,NaN,False,"broad_money_growth_pct, real_interest_rate_pct..."
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,M6,M6 - Tourism robustness,estimated,NaN,False,"broad_money_growth_pct, deposit_interest_rate_..."
7,M7_human_capital_robustnessa_from_M4_real_inte...,M7,M7 - Human capital robustness,estimated,NaN,False,"broad_money_growth_pct, real_interest_rate_pct..."
8,M7_human_capital_robustnessb_from_M2_main_mone...,M7,M7 - Human capital robustness,estimated,NaN,False,"broad_money_growth_pct, deposit_interest_rate_..."


## Load Exported Estimation Artifacts (Produced by Notebook 02)

If these files are missing, run `02_panel_diagnostics.ipynb` first.


In [96]:
def read_csv_if_exists(path: Path, **kwargs):
    if path.exists():
        return pd.read_csv(path, **kwargs)
    print('MISSING:', path.name, '(run 02_panel_diagnostics.ipynb)')
    return None

regression_table_main = read_csv_if_exists(OUTPUTS_DIR / 'regression_table_main.csv', index_col=0)
model_coefficients = read_csv_if_exists(OUTPUTS_DIR / 'model_coefficients.csv')
model_fit_stats = read_csv_if_exists(OUTPUTS_DIR / 'model_fit_stats.csv')
model_sample_audit = read_csv_if_exists(OUTPUTS_DIR / 'model_sample_audit.csv')
low_gap_interpretations = read_csv_if_exists(OUTPUTS_DIR / 'low_gap_coefficient_interpretations.csv')


In [97]:
def filter_to_estimated(frame):
    if frame is None:
        return None
    if 'model_id' not in frame.columns:
        return frame
    return frame[frame['model_id'].isin(estimated_model_ids)].copy()

model_coefficients_est = filter_to_estimated(model_coefficients)
model_fit_stats_est = filter_to_estimated(model_fit_stats)
model_sample_audit_est = filter_to_estimated(model_sample_audit)

if model_fit_stats_est is not None:
    compact_display(model_fit_stats_est, n=25)


,model_id,estimator,covariance_type,nobs,r_squared,within_r2,adj_r_squared,slope_adjusted_within_r2,r2_adj_r2_gap
0,M1_baseline_liquidity,pooled_ols,country-clustered,209.0000,0.6393,0.6393,0.6304,0.6304,0.0089
1,M1_baseline_liquidity,fixed_effects,entity-clustered,209.0000,0.0126,0.0305,-0.1941,0.0066,0.2067
2,M1_baseline_liquidity,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,209.0000,0.0126,0.0305,-0.1941,0.0066,0.2067
3,M1_baseline_liquidity,random_effects,entity-clustered,209.0000,0.1256,0.0266,0.1040,0.0026,0.0215
4,M2_main_monetary_policy,pooled_ols,country-clustered,149.0000,0.7285,0.7285,0.7170,0.7170,0.0115
5,M2_main_monetary_policy,fixed_effects,entity-clustered,149.0000,0.0388,0.0655,-0.2479,0.0260,0.2867
6,M2_main_monetary_policy,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,149.0000,0.0388,0.0655,-0.2479,0.0260,0.2867
7,M2_main_monetary_policy,random_effects,entity-clustered,149.0000,NaN,NaN,NaN,NaN,NaN
8,M3_lagged_main_model,pooled_ols,country-clustered,145.0000,0.7153,0.7153,0.7029,0.7029,0.0124
9,M3_lagged_main_model,fixed_effects,entity-clustered,145.0000,0.0669,0.0883,-0.2105,0.0486,0.2774


## Per-Model Regression Tables (Module Exports)

Notebook 02 exports raw, text-based regression output and a short text note for each workbook-estimated model:

- `{model_id}_regression_raw.txt`
- `{model_id}_regression_detail_note.txt`

Display these per-model outputs first (as raw module text), then show the consolidated summary table.


In [98]:
workbook_name_lookup = estimated_catalog.set_index('model_id')['workbook_model'].to_dict()

any_missing = False
for model_id in estimated_model_ids:
    raw_path = OUTPUTS_DIR / f'{model_id}_regression_raw.txt'
    note_path = OUTPUTS_DIR / f'{model_id}_regression_detail_note.txt'

    if not raw_path.exists() and not note_path.exists():
        any_missing = True
        continue

    workbook_name = workbook_name_lookup.get(model_id, model_id)
    display(Markdown(f"### {workbook_name}  \\n`{model_id}`"))

    if note_path.exists():
        note = note_path.read_text().strip()
        if note:
            display(Markdown('**Model note**'))
            print(note)

    if raw_path.exists():
        raw_text = raw_path.read_text().rstrip()
        if raw_text:
            print(raw_text)
        else:
            print('(empty)', raw_path.name)
    else:
        print('MISSING:', raw_path.name)

if any_missing:
    print('Some per-model regression raw outputs are missing. Re-run 02_panel_diagnostics.ipynb if needed.')


### M1 - Baseline liquidity  \n`M1_baseline_liquidity`

**Model note**

Headline estimator: Two-way FE, Driscoll-Kraay SE. Hausman status: Diagnostic only; negative statistic clipped. Recommendation: Use as baseline if muốn giữ mẫu lớn nhất.. Structural coverage note: No structural monetary-proxy coverage note. Hausman note: covariance-difference inversion became numerically unstable, so the raw statistic turned negative. The displayed statistic was clipped at zero, which mechanically gives p=1.0000.
===== pooled_ols =====
OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.639
Model:                            OLS   Adj. R-squared:                  0.630
Method:                 Least Squares   F-statistic:                     58.21
Date:                Wed, 27 May 2026   Prob (F-statistic):           1.40e-06
Time:                        15:17:32   Log-Likelihood:                -566.82
No. Observations:                 209   AIC:                             1146.
Df Residuals:     

### M2 - Main monetary policy  \n`M2_main_monetary_policy`

**Model note**

...

**Model note**

Headline estimator: Two-way FE, Driscoll-Kraay SE. Hausman status: Diagnostic only; Hausman favors FE. Recommendation: Optional after fixing number-format issue in combined file.. Structural coverage note: No structural monetary-proxy coverage note. Hausman note: positive test statistic with the standard FE-versus-RE interpretation.
===== pooled_ols =====
OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.738
Model:                            OLS   Adj. R-squared:                  0.725
Method:                 Least Squares   F-statistic:                     2.081
Date:                Wed, 27 May 2026   Prob (F-statistic):              0.197
Time:                        15:17:33   Log-Likelihood:                -395.39
No. Observations:                 151   AIC:                             806.8
Df Residuals:                     143   BIC:                             830.9
Df Model:                           7 

### M7 - Human capital robustness  \n`M7_human_capital_robustnessb_from_M2_main_monetary_policy`

**Model note**

Headline estimator: Two-way FE, Driscoll-Kraay SE. Hausman status: Diagnostic only; Hausman does not reject RE. Recommendation: Optional after fixing number-format issue in combined file.. Structural coverage note: No structural monetary-proxy coverage note. Hausman note: positive test statistic with the standard FE-versus-RE interpretation.
===== pooled_ols =====
OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.730
Model:                            OLS   Adj. R-squared:                  0.717
Method:                 Least Squares   F-statistic:                     106.1
Date:                Wed, 27 May 2026   Prob (F-statistic):           8.02e-06
Time:                        15:17:33   Log-Likelihood:                -393.04
No. Observations:                 149   AIC:                             802.1
Df Residuals:                     141   BIC:                             826.1
Df Model:                    

## Summary Table (Module Export)

Consolidated view exported by notebook 02: `outputs/regression_table_main.csv`.


In [99]:
regression_table_main if regression_table_main is not None else 'Run notebook 02 first.'


,M1 - Baseline liquidity,M2 - Main monetary policy,M3 - Lagged main model (t-1),M4 - Real interest robustness,M5 - Lending rate robustness
metric,,,,,
Broad money growth (annual %),0.0460*\n(0.0277),0.0451\n(0.0388),0.1223**\n(0.0572),0.0351\n(0.0426),0.0627*\n(0.0360)
Deposit interest rate (%),NaN,0.0922\n(0.1242),0.1274\n(0.1521),NaN,NaN
Real interest rate (%),NaN,NaN,NaN,-0.3225\n(0.2370),NaN
Lending interest rate (%),NaN,NaN,NaN,NaN,0.2500\n(0.1738)
Trade (% GDP),-0.0014\n(0.0123),-0.0154\n(0.0175),-0.0170\n(0.0198),-0.0126\n(0.0184),-0.0182\n(0.0167)
"Inflation, GDP deflator (%)",-0.0195\n(0.0576),0.0017\n(0.0441),-0.0322\n(0.0485),-0.2957\n(0.1988),-0.0200\n(0.0396)
Log GDP per capita,0.2059\n(1.5284),0.9642*\n(0.5560),1.1131\n(0.8672),-0.9521\n(1.2826),1.8815***\n(0.5928)
Exchange rate depreciation (%),0.0153\n(0.0272),0.0539\n(0.0413),0.0585\n(0.0469),0.0801*\n(0.0427),0.0624\n(0.0384)
Observations,209,149,145,164,151


## Note On Outputs

This notebook does **not** synthesize new result tables; it reads and visualizes the module exports
produced by notebook 02 under `outputs/`.


## Diagnostics (Module Outputs)

Use the consolidated exports produced by `src.estimate_and_export` (not per-model file globs).


In [100]:
model_diagnostics = read_csv_if_exists(OUTPUTS_DIR / 'model_diagnostics.csv')
model_vif = read_csv_if_exists(OUTPUTS_DIR / 'model_vif.csv')
model_panel_balance = read_csv_if_exists(OUTPUTS_DIR / 'model_panel_balance_summary.csv')

model_diagnostics_est = filter_to_estimated(model_diagnostics)
model_vif_est = filter_to_estimated(model_vif)
model_panel_balance_est = filter_to_estimated(model_panel_balance)

compact_display(model_diagnostics_est, n=12) if model_diagnostics_est is not None else 'Run notebook 02 first.'


,model_id,test_model,condition_number,breusch_pagan_lm_stat,breusch_pagan_lm_p_value,breusch_pagan_f_stat,breusch_pagan_f_p_value,white_lm_stat,white_lm_p_value,white_f_stat,white_f_p_value,pesaran_cd_stat,pesaran_cd_p_value,pesaran_cd_country_pairs,pesaran_cd_min_pair_years,pesaran_cd_max_pair_years,main_estimator,main_covariance_type,random_effects_available,re_failure_reason
0,M1_baseline_liquidity,pooled_ols,"2,148.2153",32.2765,0.0000,7.4151,0.0000,60.4232,0.0000,3.8228,0.0000,-1.7715,0.0765,45,7,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN
1,M2_main_monetary_policy,pooled_ols,"3,753.0678",42.7953,0.0000,9.5365,0.0000,50.8382,0.0036,2.3210,0.0010,-0.8852,0.3760,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,False,float division by zero
2,M3_lagged_main_model,pooled_ols,"3,837.7614",39.5946,0.0000,8.6397,0.0000,58.5116,0.0004,2.9316,0.0000,-0.4329,0.6651,21,17,22,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,False,float division by zero
3,M4_real_interest_robustness,pooled_ols,"2,858.3106",47.2362,0.0000,10.5856,0.0000,75.2246,0.0000,4.2682,0.0000,-1.9275,0.0539,28,9,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN
4,M5_lending_rate_robustness,pooled_ols,"3,493.4125",43.9226,0.0000,9.8447,0.0000,52.1806,0.0025,2.4055,0.0006,-0.9145,0.3604,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,False,float division by zero
5,M6_tourism_robustnessa_from_M4_real_interest_r...,pooled_ols,"3,614.1359",42.9005,0.0000,8.1073,0.0000,54.9065,0.0173,1.8774,0.0068,-0.6381,0.5234,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,pooled_ols,"3,786.8355",41.4088,0.0000,7.7524,0.0000,53.5829,0.0231,1.8131,0.0102,-0.8359,0.4032,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN
7,M7_human_capital_robustnessa_from_M4_real_inte...,pooled_ols,"4,514.3748",42.8943,0.0000,8.1057,0.0000,54.1120,0.0206,1.8351,0.0088,-0.7483,0.4543,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN
8,M7_human_capital_robustnessb_from_M2_main_mone...,pooled_ols,"4,369.3459",42.6433,0.0000,8.0762,0.0000,55.7404,0.0144,1.9297,0.0051,-0.8941,0.3713,21,17,23,fixed_effects_driscoll_kraay,Driscoll-Kraay kernel Bartlett,True,NaN


## Summary Figures (PNG Only)

Produce two compact figures:

- Sample sizes by model
- Key coefficients across models (with CI when available)


In [101]:
def preferred_estimator_per_model(fit_stats):
    if fit_stats is None or fit_stats.empty:
        return {}
    preferred = {}
    for model_id, grp in fit_stats.groupby('model_id'):
        if grp['estimator'].astype(str).str.contains('random_effects').any():
            preferred[model_id] = 'random_effects'
        elif grp['estimator'].astype(str).str.contains('fixed_effects').any():
            preferred[model_id] = 'fixed_effects'
        else:
            preferred[model_id] = grp['estimator'].astype(str).iloc[0]
    return preferred

def load_sample_fit_summary_fallback() -> pd.DataFrame:
    rows = []
    for path in sorted(OUTPUTS_DIR.glob('*_sample_fit_summary.csv')):
        tmp = pd.read_csv(path)
        if 'model_id' not in tmp.columns:
            tmp = tmp.assign(model_id=path.name.replace('_sample_fit_summary.csv', ''))
        rows.append(tmp)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

def sample_sizes_table(fit_stats, estimated_ids, model_order):
    if fit_stats is not None and not fit_stats.empty and {'model_id', 'nobs'}.issubset(fit_stats.columns):
        preferred_map = preferred_estimator_per_model(fit_stats)
        tmp = fit_stats.copy()
        tmp = tmp[tmp['model_id'].isin(estimated_ids)].copy()
        if 'estimator' in tmp.columns and preferred_map:
            tmp = tmp[tmp.apply(lambda row: str(row['estimator']) == preferred_map.get(row['model_id'], str(row['estimator'])), axis=1)].copy()
        out = (
            tmp.groupby('model_id', as_index=False)['nobs']
            .max()
            .rename(columns={'nobs': 'nobs'})
        )
        out['nobs'] = out['nobs'].astype(float).round().astype(int)
    else:
        fallback = load_sample_fit_summary_fallback()
        if fallback.empty or 'nobs' not in fallback.columns:
            return pd.DataFrame(columns=['model_id', 'nobs'])
        out = fallback[['model_id', 'nobs']].copy()
        out = out[out['model_id'].isin(estimated_ids)].copy()
        out['nobs'] = out['nobs'].astype(float).round().astype(int)

    out['model_id'] = out['model_id'].astype(str)
    order_index = {m: i for i, m in enumerate(model_order)}
    out['order'] = out['model_id'].map(order_index).fillna(1e9)
    out = out.sort_values(['order', 'model_id']).drop(columns=['order']).reset_index(drop=True)
    return out

model_order = estimated_catalog['model_id'].tolist()
nobs_table = sample_sizes_table(model_fit_stats_est, estimated_model_ids, model_order)
nobs_table


,model_id,nobs
0,M1_baseline_liquidity,209
1,M2_main_monetary_policy,149
2,M3_lagged_main_model,145
3,M4_real_interest_robustness,164
4,M5_lending_rate_robustness,151
5,M6_tourism_robustnessa_from_M4_real_interest_r...,151
6,M6_tourism_robustnessb_from_M2_main_monetary_p...,149
7,M7_human_capital_robustnessa_from_M4_real_inte...,151
8,M7_human_capital_robustnessb_from_M2_main_mone...,149


In [102]:
if not nobs_table.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(nobs_table['model_id'], nobs_table['nobs'], color='steelblue')
    ax.set_title('Sample sizes (nobs) by model')
    ax.set_xlabel('Model')
    ax.set_ylabel('Observations (nobs)')
    for tick in ax.get_xticklabels():
        tick.set_rotation(45)
        tick.set_ha('right')
    fig.tight_layout()
    out_path = FIGURES_DIR / 'methodology__sample_sizes_by_model.png'
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    print('Saved', out_path)
else:
    print('No sample-size information available (missing model_fit_stats + *_sample_fit_summary.csv).')


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/figures/methodology__sample_sizes_by_model.png


/var/folders/zg/_wfh70cn569_5p6d24bj1_tr0000gn/T/ipykernel_16869/2636981638.py:10: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


In [103]:
def key_terms_present(coeff_df: pd.DataFrame, requested: list[str]) -> list[str]:
    present = set(coeff_df['term'].astype(str).unique())
    return [term for term in requested if term in present]

requested_terms = [
    'broad_money_growth_pct',
    'inflation_gdp_deflator_pct',
    'trade_pct_gdp',
    'ln_gdppc',
    'xr_dep_pct',
    # optional / robustness terms
    'real_interest_rate_pct',
    'deposit_interest_rate_pct',
    'lending_interest_rate_pct',
    'ln_tourism_arrivals',
    'hc_human_capital_index',
]

if model_coefficients_est is None or model_coefficients_est.empty:
    print('No model_coefficients available (run 02_panel_diagnostics.ipynb).')
else:
    preferred_fit = preferred_estimator_per_model(model_fit_stats_est) if model_fit_stats_est is not None else {}
    coeff = model_coefficients_est.copy()
    if preferred_fit and 'estimator' in coeff.columns:
        coeff = coeff[coeff.apply(lambda row: str(row['estimator']) == preferred_fit.get(row['model_id'], str(row['estimator'])), axis=1)].copy()

    terms = key_terms_present(coeff, requested_terms)
    if not terms:
        raise ValueError('None of the requested key terms are present in model_coefficients.csv.')

    coeff = coeff[coeff['term'].isin(terms)].copy()
    coeff['term_label'] = coeff['term'].map(lambda t: VARIABLE_LABELS.get(t, t))
    order_index = {m: i for i, m in enumerate(model_order)}
    coeff['model_order'] = coeff['model_id'].map(order_index).fillna(1e9)
    coeff = coeff.sort_values(['term', 'model_order', 'model_id']).reset_index(drop=True)
    coeff.head(10)


In [104]:
if model_coefficients_est is not None and not model_coefficients_est.empty:
    fig, ax = plt.subplots(figsize=(13, 5))

    # Scatter + CI by term (small number of terms and models, so a single axis is readable).
    x_positions = {m: i for i, m in enumerate(model_order)}
    x = coeff['model_id'].map(x_positions)

    term_labels = list(dict.fromkeys(coeff['term_label'].tolist()))
    palette = sns.color_palette('tab10', n_colors=max(3, len(term_labels)))
    color_map = {label: palette[i % len(palette)] for i, label in enumerate(term_labels)}

    for label in term_labels:
        tmp = coeff[coeff['term_label'].eq(label)]
        xs = tmp['model_id'].map(x_positions).astype(float).values
        ys = tmp['coef'].astype(float).values
        ax.scatter(xs, ys, label=label, color=color_map[label], s=45)
        if {'ci_low', 'ci_high'}.issubset(tmp.columns):
            yerr_low = ys - tmp['ci_low'].astype(float).values
            yerr_high = tmp['ci_high'].astype(float).values - ys
            ax.errorbar(xs, ys, yerr=[yerr_low, yerr_high], fmt='none', ecolor=color_map[label], alpha=0.55, linewidth=1)

    ax.axhline(0, color='black', linewidth=1, alpha=0.6)
    ax.set_xticks(list(x_positions.values()))
    ax.set_xticklabels(list(x_positions.keys()), rotation=45, ha='right')
    ax.set_title('Key coefficients across workbook-estimated models (preferred estimator)')
    ax.set_xlabel('Model')
    ax.set_ylabel('Coefficient (with 95% CI when available)')
    ax.legend(loc='best', frameon=True, fontsize=9)

    fig.tight_layout()
    out_path = FIGURES_DIR / 'methodology__key_coefficients_across_models.png'
    fig.savefig(out_path, dpi=220)
    plt.close(fig)
    print('Saved', out_path)


/var/folders/zg/_wfh70cn569_5p6d24bj1_tr0000gn/T/ipykernel_16869/979919308.py:30: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  fig.tight_layout()


Saved /Users/bunnypro/Projects/monetary_policy_fdi_analysis/outputs/figures/methodology__key_coefficients_across_models.png


## Optional: Interpretation Tables (If Present)


In [105]:
compact_display(model_sample_audit_est, n=25) if model_sample_audit_est is not None else 'Run notebook 02 first.'


,comparison_id,base_model_id,added_model_id,base_rows,added_rows,rows_lost_from_base,base_countries,added_countries,countries_lost_from_base,lost_countries,top_loss_drivers
0,M1_to_M2,M1_baseline_liquidity,M2_main_monetary_policy,209,149,60,10,7,3,"Brunei Darussalam, Cambodia, Lao PDR, Philippi...",deposit_interest_rate_pct: 27
1,M2_to_M3_lagged,M2_main_monetary_policy,M3_lagged_main_model,149,145,8,7,7,0,"Brunei Darussalam, Indonesia, Malaysia, Philip...",xr_dep_pct_lag1: 6; deposit_interest_rate_pct_...
2,M2_to_M5_lending,M2_main_monetary_policy,M5_lending_rate_robustness,149,151,0,7,7,0,NaN,NaN
3,M2_to_M6_tourism,M2_main_monetary_policy,M6_tourism_robustnessb_from_M2_main_monetary_p...,149,149,0,7,7,0,NaN,NaN
4,M4_to_M6_tourism,M4_real_interest_robustness,M6_tourism_robustnessa_from_M4_real_interest_r...,164,151,13,8,7,1,Timor-Leste,ln_tourism_arrivals: 13
5,M2_to_M7_hc,M2_main_monetary_policy,M7_human_capital_robustnessb_from_M2_main_mone...,149,149,0,7,7,0,NaN,NaN
6,M4_to_M7_hc,M4_real_interest_robustness,M7_human_capital_robustnessa_from_M4_real_inte...,164,151,13,8,7,1,Timor-Leste,hc_human_capital_index: 13


In [106]:
compact_display(low_gap_interpretations, n=25) if low_gap_interpretations is not None else 'Run notebook 02 first.'


,gap_rank,model_id,model,estimator,term,base_term,coefficient_label,coef,p_value,coefficient_direction,significance,expected_sign,expected_sign_source,sign_alignment,interpretation
0,1,M1_baseline_liquidity,M1 - Baseline liquidity,"Two-way FE, Driscoll-Kraay SE",broad_money_growth_pct,broad_money_growth_pct,Broad money growth (annual %),0.0460,0.0994,positive,marginally significant at 10%,positive,workbook Variable selection expected_sign,matches expected positive sign,Broad money growth (annual %): coefficient 0.0...
1,1,M1_baseline_liquidity,M1 - Baseline liquidity,"Two-way FE, Driscoll-Kraay SE",inflation_gdp_deflator_pct,inflation_gdp_deflator_pct,"Inflation, GDP deflator (%)",-0.0195,0.7352,negative,not statistically significant at 10%,negative,workbook Variable selection expected_sign,matches expected negative sign,"Inflation, GDP deflator (%): coefficient -0.01..."
2,1,M1_baseline_liquidity,M1 - Baseline liquidity,"Two-way FE, Driscoll-Kraay SE",trade_pct_gdp,trade_pct_gdp,Trade (% GDP),-0.0014,0.9104,negative,not statistically significant at 10%,positive,workbook Variable selection expected_sign,does not match expected positive sign,Trade (% GDP): coefficient -0.0014 is negative...
3,1,M1_baseline_liquidity,M1 - Baseline liquidity,"Two-way FE, Driscoll-Kraay SE",ln_gdppc,ln_gdppc,Log GDP per capita,0.2059,0.8930,positive,not statistically significant at 10%,positive,workbook Variable selection expected_sign,matches expected positive sign,Log GDP per capita: coefficient 0.2059 is posi...
4,1,M1_baseline_liquidity,M1 - Baseline liquidity,"Two-way FE, Driscoll-Kraay SE",xr_dep_pct,xr_dep_pct,Exchange rate depreciation (%),0.0153,0.5743,positive,not statistically significant at 10%,ambiguous,workbook Variable selection expected_sign,no directional sign test because theory/workbo...,Exchange rate depreciation (%): coefficient 0....
5,2,M4_real_interest_robustness,M4 - Real interest robustness,"Two-way FE, Driscoll-Kraay SE",broad_money_growth_pct,broad_money_growth_pct,Broad money growth (annual %),0.0351,0.4112,positive,not statistically significant at 10%,positive,workbook Variable selection expected_sign,matches expected positive sign,Broad money growth (annual %): coefficient 0.0...
6,2,M4_real_interest_robustness,M4 - Real interest robustness,"Two-way FE, Driscoll-Kraay SE",real_interest_rate_pct,real_interest_rate_pct,Real interest rate (%),-0.3225,0.1761,negative,not statistically significant at 10%,negative,workbook Variable selection expected_sign,matches expected negative sign,Real interest rate (%): coefficient -0.3225 is...
7,2,M4_real_interest_robustness,M4 - Real interest robustness,"Two-way FE, Driscoll-Kraay SE",inflation_gdp_deflator_pct,inflation_gdp_deflator_pct,"Inflation, GDP deflator (%)",-0.2957,0.1392,negative,not statistically significant at 10%,negative,workbook Variable selection expected_sign,matches expected negative sign,"Inflation, GDP deflator (%): coefficient -0.29..."
8,2,M4_real_interest_robustness,M4 - Real interest robustness,"Two-way FE, Driscoll-Kraay SE",trade_pct_gdp,trade_pct_gdp,Trade (% GDP),-0.0126,0.4924,negative,not statistically significant at 10%,positive,workbook Variable selection expected_sign,does not match expected positive sign,Trade (% GDP): coefficient -0.0126 is negative...
9,2,M4_real_interest_robustness,M4 - Real interest robustness,"Two-way FE, Driscoll-Kraay SE",ln_gdppc,ln_gdppc,Log GDP per capita,-0.9521,0.4592,negative,not statistically significant at 10%,positive,workbook Variable selection expected_sign,does not match expected positive sign,Log GDP per capita: coefficient -0.9521 is neg...
